<a href="https://colab.research.google.com/github/sunnysavita10/Indepth-GENAI/blob/main/RAG_With_Knowledge_graph(Neo4j).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install --upgrade --quiet  langchain langchain-community langchain-openai langchain-experimental neo4j wikipedia tiktoken yfiles_jupyter_graphs

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate
from typing import Tuple, List, Optional
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import ConfigurableField
from yfiles_jupyter_graphs import GraphWidget
from neo4j import GraphDatabase

In [5]:
# from google.colab import userdata
OPENAI_API_KEY=""

In [10]:
import os

In [11]:
try:
  import google.colab
  from google.colab import output
  output.enable_custom_widget_manager()
except:
  pass

In [12]:
from langchain_community.vectorstores import Neo4jVector

In [11]:
# from google.colab import userdata
# OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')

In [13]:
NEO4J_URI="neo4j+s://03a72e82.databases.neo4j.io"
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD="SqVXPboq2rbbrVjAQLLDD-KwqxIhnmqhyJ5tqU3GKdk"


In [14]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

In [15]:
from langchain_community.graphs import Neo4jGraph

In [16]:
graph = Neo4jGraph()

In [17]:
file_path = "/home/aura/gsoc/wbm_ai_kg/PageExtraction/channel/20240729191002/content.txt"

with open(file_path, 'r', encoding='utf-8') as file:
    content = file.read()

# If your previous code expects `raw_documents` to be a list of documents
raw_documents = [{"page_content": content, "meta": {}}]

In [19]:
raw_documents

[{'page_content': '17-year-old Ali Al-Shakati arrested in connection with the stabbings in Southport, England\n17-year-old Ali-Al-Shakati from Banks arrested in connection with the stabbings in Southport, England. He was on MI6 watch list and was known to Liverpool mental health services. He was an asylum seeker who came to UK by boat last year.\n17-year-old Ali Al-Shakati arrested in connection with the stabbings in Southport, England News World Entertainment Sports Business Politics Technology Weather Connect with us Channel3 NOW 17-year-old Ali Al-Shakati arrested in connection with the stabbings in Southport, England News 17-year-old Ali Al-Shakati arrested in connection with the stabbings in Southport, England 22-year-old Quincy Jones killed after shooting in Bridgeport Township UPDATE: 2 dead in mass shooting at Maplewood Park in Rochester, victims identified Victims identified: 3 killed after pontoon boat capsized on Lake Powell in northern Arizona 25-year-old Tyasia Manning kil

In [20]:
len(raw_documents)

1

In [25]:
raw_documents[:5]
# content = raw_documents[0]["page_content"]

# # Split the content directly using the text splitter
# split_content = text_splitter.split_text(content)

# # Create the document structure from the split content
# documents = [{"page_content": chunk, "meta": {}} for chunk in split_content]

In [29]:
# from langchain.text_splitter import TokenTextSplitter
# text_splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=24)
# documents = text_splitter.split_documents(raw_documents[:5])
from langchain.schema import Document

raw_documents = [Document(page_content=content, metadata={})]

documents = text_splitter.split_documents(raw_documents[:5])


In [30]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo-0125")

In [31]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
llm_transformer = LLMGraphTransformer(llm=llm)

In [32]:
graph_documents = llm_transformer.convert_to_graph_documents(documents)

In [33]:
graph_documents

[GraphDocument(nodes=[Node(id='Ali Al-Shakati', type='Person'), Node(id='Southport', type='Place'), Node(id='England', type='Place'), Node(id='Mi6', type='Organization'), Node(id='Liverpool Mental Health Services', type='Organization'), Node(id='Banks', type='Place'), Node(id='Asylum Seeker', type='Person')], relationships=[Relationship(source=Node(id='Ali Al-Shakati', type='Person'), target=Node(id='Southport', type='Place'), type='ARRESTED_IN'), Relationship(source=Node(id='Ali Al-Shakati', type='Person'), target=Node(id='England', type='Place'), type='ARRESTED_IN'), Relationship(source=Node(id='Ali Al-Shakati', type='Person'), target=Node(id='Mi6', type='Organization'), type='ON_WATCH_LIST'), Relationship(source=Node(id='Ali Al-Shakati', type='Person'), target=Node(id='Liverpool Mental Health Services', type='Organization'), type='KNOWN_TO'), Relationship(source=Node(id='Ali Al-Shakati', type='Person'), target=Node(id='Banks', type='Place'), type='FROM'), Relationship(source=Node(id

In [34]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

In [40]:

default_cypher = "MATCH (s)-[r]->(t) WHERE NOT s:Document AND NOT t:Document RETURN s,r,t LIMIT 100"


In [41]:
from yfiles_jupyter_graphs import GraphWidget
from neo4j import GraphDatabase

In [42]:
try:
  import google.colab
  from google.colab import output
  output.enable_custom_widget_manager()
except:
  pass

In [43]:
def showGraph(cypher: str = default_cypher):
    # create a neo4j session to run queries
    driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"]))
    session = driver.session()
    widget = GraphWidget(graph = session.run(cypher).graph())
    widget.node_label_mapping = 'id'
    display(widget)
    return widget

In [44]:
showGraph()

GraphWidget(layout=Layout(height='800px', width='100%'))

GraphWidget(layout=Layout(height='800px', width='100%'))

In [45]:
from typing import Tuple, List, Optional

In [46]:
from langchain_community.vectorstores import Neo4jVector

In [47]:
from langchain_openai import OpenAIEmbeddings
vector_index = Neo4jVector.from_existing_graph(
    OpenAIEmbeddings(),
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding"
)

In [48]:
graph.query("CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]")

[]

In [49]:
from langchain_core.pydantic_v1 import BaseModel, Field
# Extract entities from text
class Entities(BaseModel):
    """Identifying information about entities."""

    names: List[str] = Field(
        ...,
        description="All the person, organization, or business entities that "
        "appear in the text",
    )


In [50]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

In [51]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following "
            "input: {question}",
        ),
    ]
)

In [52]:
entity_chain = prompt | llm.with_structured_output(Entities)

In [53]:
entity_chain.invoke({"question": "Who is ali?"}).names

['Ali']

In [54]:
from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars

In [55]:
def generate_full_text_query(input: str) -> str:
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()


In [56]:
# Fulltext index query
def structured_retriever(question: str) -> str:
    result = ""
    entities = entity_chain.invoke({"question": question})
    for entity in entities.names:
        response = graph.query(
            """CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})
            YIELD node,score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )
        result += "\n".join([el['output'] for el in response])
    return result

In [57]:
print(structured_retriever("Who is Ali?"))

Ali Al-Shakati - ARRESTED_IN -> Southport
Ali Al-Shakati - ARRESTED_IN -> England
Ali Al-Shakati - ON_WATCH_LIST -> Mi6
Ali Al-Shakati - KNOWN_TO -> Liverpool Mental Health Services
Ali Al-Shakati - FROM -> Banks
Ali Al-Shakati - IS -> Asylum Seeker
Ali Al-Shakati - ARRESTED_IN_CONNECTION_WITH -> Southport, England
Ali Al-Shakati - ON_WATCH_LIST_AND_KNOWN_TO -> Mi6
Ali Al-Shakati - ARRIVED_IN_LAST_YEAR -> Uk
Resources - INCLUDING -> Air Ambulance


In [58]:
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [el.page_content for el in vector_index.similarity_search(question)]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{"#Document ". join(unstructured_data)}
    """
    return final_data

In [59]:
_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question,
in its original language.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

In [60]:
CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(_template)

In [61]:
def _format_chat_history(chat_history: List[Tuple[str, str]]) -> List:
    buffer = []
    for human, ai in chat_history:
        buffer.append(HumanMessage(content=human))
        buffer.append(AIMessage(content=ai))
    return buffer

In [62]:
_search_query = RunnableBranch(
    # If input includes chat_history, we condense it with the follow-up question
    (
        RunnableLambda(lambda x: bool(x.get("chat_history"))).with_config(
            run_name="HasChatHistoryCheck"
        ),  # Condense follow-up question and chat into a standalone_question
        RunnablePassthrough.assign(
            chat_history=lambda x: _format_chat_history(x["chat_history"])
        )
        | CONDENSE_QUESTION_PROMPT
        | ChatOpenAI(temperature=0)
        | StrOutputParser(),
    ),
    # Else, we have no chat history, so just pass through the question
    RunnableLambda(lambda x : x["question"]),
)

In [63]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""

In [64]:
prompt = ChatPromptTemplate.from_template(template)

In [65]:
chain = (
    RunnableParallel(
        {
            "context": _search_query | retriever,
            "question": RunnablePassthrough(),
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [66]:
chain.invoke({"question": "What did Ali do?"})

Search query: What did Ali do?


'Ali Al-Shakati was arrested in connection with the stabbings in Southport, England.'

In [67]:
chain.invoke(
    {
        "question": "What happened in Southport",
        "chat_history": [("What did Ali do?", "Ali Al-Shakati was arrested in connection with the stabbings in Southport, England.")],
    }
)

Search query: What happened in Southport?


'A stabbing incident occurred in Southport, England, resulting in multiple injuries and fatalities. Ali Al-Shakati was arrested in connection with the stabbings. The incident is not terror-related.'

In [68]:
chain.invoke(
    {
        "question":"Give the brief of the news"
    }
)

Search query: Give the brief of the news


'The news is about a stabbing incident in Southport, England where a 17-year-old named Ali Al-Shakati, who was on the MI6 watch list and known to mental health services, has been arrested. Multiple people were attacked, including children and adults, with at least one fatality reported. The incident is not terror-related, and investigations are ongoing.'

In [69]:
chain.invoke(
    {
        "question":"What did MI6 do?"
    }
)

Search query: What did MI6 do?


Failed to write data to connection ResolvedIPv4Address(('34.126.161.242', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687)))
Failed to write data to connection IPv4Address(('03a72e82.databases.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687)))
Failed to write data to connection ResolvedIPv4Address(('34.126.161.242', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687)))
Failed to write data to connection IPv4Address(('03a72e82.databases.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687)))


'MI6 had Ali Al-Shakati on their watch list and knew about him.'